# 🌳 03 — Change Detection
### Amazon Deforestation Change Detection Pipeline

Runs **DOFA** segmentation on both scenes, then compares them pixel-by-pixel to produce:
- **Binary change map** — which pixels changed
- **Class-level change map** — what each pixel changed from/to
- **Change summary CSV** — hectares lost per class transition

---
**Pipeline:**
1. ✅ `01_download_data.ipynb`
2. ✅ `02_prepare_scenes.ipynb`
3. 🌳 `03_detect_changes.ipynb` ← *you are here*
4. 🎨 `04_visualise.ipynb`


## ⚙️ Configuration
> ⚠️ Update `CHECKPOINT_PATH` to point to your trained DOFA checkpoint.

In [ ]:
import numpy as np
import torch
import rasterio
import pandas as pd
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import sys, warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path(".").resolve()))

# ── Update this path to your checkpoint ──────────────────────────────────────
CHECKPOINT_PATH = Path(
    "logs/gdl_experiment/version_11/checkpoints/model-epoch=00-val_loss=0.141.ckpt"
)

PREPARED_DIR = Path("data/prepared")
OUTPUT_DIR   = Path("outputs/change")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATCH_SIZE          = 64
WAVELENGTHS         = torch.tensor([0.665, 0.549, 0.481])  # R, G, B in micrometres
MEAN                = [0.485, 0.456, 0.406]
STD                 = [0.229, 0.224, 0.225]
PIXEL_SIZE_M        = 10.0
HECTARES_PER_PIXEL  = (PIXEL_SIZE_M ** 2) / 10_000

CLASS_NAMES = [
    "Annual Crop", "Forest", "Herbaceous Veg", "Highway",
    "Industrial", "Pasture", "Permanent Crop", "Residential",
    "River", "Sea / Lake",
]
NUM_CLASSES = len(CLASS_NAMES)
print("✅ Configuration set")
print(f"   Checkpoint: {CHECKPOINT_PATH}")
print(f"   Checkpoint exists: {CHECKPOINT_PATH.exists()}")


## 📦 Step 1 — Load DOFA Model

In [ ]:
import segmentation_models_pytorch as smp
from geo_deep_learning.tasks_with_models.segmentation_dofa import SegmentationDOFA

print(f"Loading checkpoint from:\n  {CHECKPOINT_PATH}")
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
hp   = ckpt["hyper_parameters"]

model = SegmentationDOFA(
    encoder       = hp["encoder"],
    pretrained    = False,
    image_size    = tuple(hp["image_size"]),
    num_classes   = hp["num_classes"],
    max_samples   = hp.get("max_samples", 2),
    loss          = smp.losses.DiceLoss(mode="multiclass", from_logits=True),
    class_labels  = hp.get("class_labels"),
    class_colors  = hp.get("class_colors"),
    freeze_layers = hp.get("freeze_layers"),
)
model.configure_model()
state = {k.removeprefix("model."): v for k, v in ckpt["state_dict"].items()}
model.model.load_state_dict(state, strict=True)
model.eval()

total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ Model loaded")
print(f"   Total params    : {total/1e6:.1f}M")
print(f"   Trainable params: {trainable/1e6:.1f}M")


## 🗺️ Step 2 — Load Scenes

In [ ]:
def load_scene(year):
    path = PREPARED_DIR / f"{year}_rgb.tif"
    with rasterio.open(path) as src:
        return src.read(), src.profile

scene_2019, profile = load_scene("2019")
scene_2024, _       = load_scene("2024")

# Crop to minimum common size
min_h = min(scene_2019.shape[1], scene_2024.shape[1])
min_w = min(scene_2019.shape[2], scene_2024.shape[2])
scene_2019 = scene_2019[:, :min_h, :min_w]
scene_2024 = scene_2024[:, :min_h, :min_w]

print(f"Scene shape: {scene_2019.shape}  (channels, height, width)")


## 🔍 Step 3 — Run DOFA Inference on Both Scenes
Uses a sliding patch window with 50% overlap for smooth results.

In [ ]:
tf = T.Compose([T.Resize((PATCH_SIZE, PATCH_SIZE)), T.Normalize(mean=MEAN, std=STD)])

def segment_scene(scene_array, label=""):
    _, H, W  = scene_array.shape
    conf_map = np.zeros((H, W, NUM_CLASSES), dtype=np.float32)
    count    = np.zeros((H, W), dtype=np.float32)
    stride   = PATCH_SIZE // 2
    patches_run = 0

    with torch.no_grad():
        for y in range(0, H - PATCH_SIZE + 1, stride):
            for x in range(0, W - PATCH_SIZE + 1, stride):
                patch    = scene_array[:, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                patch_pil = Image.fromarray(
                    (patch.transpose(1,2,0)*255).clip(0,255).astype(np.uint8))
                tensor   = tf(T.ToTensor()(patch_pil)).unsqueeze(0)
                wv       = WAVELENGTHS.unsqueeze(0)
                output   = model(tensor, wv)
                probs    = output.out.softmax(dim=1).squeeze(0).permute(1,2,0).numpy()
                conf_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += probs
                count[y:y+PATCH_SIZE, x:x+PATCH_SIZE]    += 1.0
                patches_run += 1
            if y % (stride * 10) == 0:
                print(f"   {label}  row {y}/{H}", end="\r")

    conf_map /= np.maximum(count, 1.0)[:, :, np.newaxis]
    seg_map   = conf_map.argmax(axis=2).astype(np.int32)
    print(f"   ✅ {label}: {patches_run} patches processed")
    return seg_map, conf_map

print("🔍 Segmenting 2019 scene...")
seg_2019, conf_2019 = segment_scene(scene_2019, "2019")

print("\n🔍 Segmenting 2024 scene...")
seg_2024, conf_2024 = segment_scene(scene_2024, "2024")


## 🔄 Step 4 — Compute Change Maps

In [ ]:
# Binary: 1 = changed, 0 = unchanged
binary_change = (seg_2019 != seg_2024).astype(np.uint8)

# Class-level: from_class*100 + to_class  (e.g. Forest→Crop = 1*100+0 = 100)
class_change  = (seg_2019 * 100 + seg_2024).astype(np.int32)

total_changed = binary_change.sum()
total_pixels  = binary_change.size
ha_changed    = total_changed * HECTARES_PER_PIXEL

print(f"📊 Change Summary")
print(f"   Pixels changed : {total_changed:,} / {total_pixels:,}  ({total_changed/total_pixels*100:.1f}%)")
print(f"   Area changed   : {ha_changed:,.0f} hectares")


## 📋 Step 5 — Class Transition Table

In [ ]:
changed_pixels = np.where(binary_change == 1)
from_cls_arr   = seg_2019[changed_pixels]
to_cls_arr     = seg_2024[changed_pixels]

records = []
for fc in range(NUM_CLASSES):
    for tc in range(NUM_CLASSES):
        if fc == tc: continue
        n = ((from_cls_arr == fc) & (to_cls_arr == tc)).sum()
        if n > 0:
            records.append({
                "from_class": CLASS_NAMES[fc], "to_class": CLASS_NAMES[tc],
                "pixels": int(n),
                "hectares": round(n * HECTARES_PER_PIXEL, 2),
                "pct_of_scene": round(n / total_pixels * 100, 3),
            })

change_df = pd.DataFrame(records).sort_values("hectares", ascending=False)
print("Top 10 class transitions:\n")
display(change_df.head(10))


## 💾 Step 6 — Save Outputs

In [ ]:
def save_raster(array, path, profile, dtype=None):
    if array.ndim == 2:
        array = array[np.newaxis, :]
    p = profile.copy()
    p.update(count=array.shape[0], dtype=dtype or str(array.dtype),
             driver="GTiff", compress="lzw")
    with rasterio.open(path, "w", **p) as dst:
        dst.write(array)
    print(f"   Saved: {path}")

save_raster(seg_2019,      OUTPUT_DIR / "segmentation_2019.tif", profile, "int32")
save_raster(seg_2024,      OUTPUT_DIR / "segmentation_2024.tif", profile, "int32")
save_raster(binary_change, OUTPUT_DIR / "binary_change.tif",     profile, "uint8")
save_raster(class_change,  OUTPUT_DIR / "class_change.tif",      profile, "int32")

csv_path = OUTPUT_DIR / "change_summary.csv"
change_df.to_csv(csv_path, index=False)
print(f"   Saved: {csv_path}")
print("\n✅ Done. Proceed to: 04_visualise.ipynb")
